### Import libraries/functions

In [ ]:
from functions.get_DLRbiomass import *
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

### DLR-Timeseries

In [ ]:
years = [2017, 2018, 2019, 2020, 2021]
arrays = []
dlr_catalog = Client.open("https://geoservice.dlr.de/eoc/ogc/stac/v1")

for y in years:
    ds = get_biomass_fromDLR(year=y, catalog=dlr_catalog)
    arrays.append(ds.read(1))
    ds.close()
stack = np.stack(arrays, axis=0)

annual_means_AGBD = np.array([np.nanmean(np.where(stack[i] > 0, stack[i], np.nan)) for i in range(len(years))])

### Convert Mg/ha to Mg

In [ ]:
area = gpd.read_file("/home/berberic/Documents/data/training_data2/bonn_trainingdata/bonn_forest.geojson").area / 10000
annual_means_AGB = annual_means_AGBD * float(area.iloc[0])

In [ ]:
print(f"Biomass loss from 2017 to 2021: {annual_means_AGB[0]-annual_means_AGB[-1]}")

plt.figure(figsize=(8, 4))
plt.plot(years, annual_means_AGB, marker="o", linestyle="-", color="green", linewidth=2)

plt.title("AGB Time Series")
plt.ylabel("AGB in Mg")
plt.xticks(years)
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()